# 03 — Silver business and data-quality rules

Run metadata-driven checks against the schema-conformed Silver tables.
Primary-key and foreign-key rules come from `schema_definition.csv`; additional
date and numeric rules come from `dq_rule_definition.csv`.

Rejected-row logging stores only key references, not complete child records.

In [ ]:
SILVER_SCHEMA = "silver"
SCHEMA_CSV_PATH = "Files/cfg_files/schema_definition.csv"
DQ_RULE_CSV_PATH = "Files/cfg_files/dq_rule_definition.csv"
MAX_REJECT_REFERENCES_PER_RULE = 100
FAIL_ON_CRITICAL = True

In [ ]:
import re, uuid
from datetime import datetime
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()

def qident(value):
    return "`" + str(value).replace("`", "``") + "`"

def silver_table(schema_name, table_name):
    return f"{SILVER_SCHEMA}.slv_{schema_name.lower()}_{table_name.lower()}"

def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS monitoring")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_pipeline_run (
  run_id STRING, pipeline_name STRING, layer STRING, source_kind STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, status STRING,
  tables_succeeded INT, tables_failed INT, rows_read BIGINT, rows_written BIGINT,
  error_message STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_table_load_metric (
  run_id STRING, layer STRING, source_kind STRING, source_object STRING,
  target_object STRING, rows_read BIGINT, rows_written BIGINT,
  duplicate_key_count BIGINT, null_primary_key_count BIGINT,
  recorded_at TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_data_quality_result (
  run_id STRING, rule_id STRING, severity STRING, rule_type STRING,
  source_table STRING, column_name STRING, status STRING,
  failed_row_count BIGINT, checked_row_count BIGINT, failure_percentage DOUBLE,
  sample_key_json STRING, checked_at TIMESTAMP, message STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_rejected_row (
  run_id STRING, rule_id STRING, source_table STRING,
  business_key_json STRING, rejection_reason STRING, rejected_at TIMESTAMP
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_referential_exception (
  run_id STRING, rule_id STRING, child_table STRING, child_column STRING,
  child_key STRING, parent_table STRING, parent_column STRING,
  detected_at TIMESTAMP
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_data_quality_rule (
  rule_id STRING, active STRING, severity STRING, rule_type STRING,
  source_schema STRING, table_name STRING, column_name STRING,
  referenced_schema STRING, referenced_table STRING, referenced_column STRING,
  operator STRING, rule_value STRING, description STRING, loaded_at TIMESTAMP
) USING DELTA
""")
append_rows("monitoring.cfg_pipeline_run", [(RUN_ID, "03_silver_business_rules", "SILVER", "LATEST",
    STARTED_AT, None, "RUNNING", 0, 0, 0, 0, None)],
    "run_id string,pipeline_name string,layer string,source_kind string,started_at timestamp,ended_at timestamp,status string,tables_succeeded int,tables_failed int,rows_read long,rows_written long,error_message string")

In [ ]:
schema_df = (spark.read.format("csv").option("header", "true").load(SCHEMA_CSV_PATH))
custom_rules_df = (spark.read.format("csv").option("header", "true").load(DQ_RULE_CSV_PATH))
schema_rows = [r.asDict() for r in schema_df.collect()]
rules = []

# Contract-generated PK completeness and table-level uniqueness checks.
primary_keys = {}
for row in schema_rows:
    if (row.get("is_primary_key") or "").upper() == "YES":
        primary_keys.setdefault((row["schema_name"], row["table_name"]), []).append(row["column_name"])
        base = {"source_schema": row["schema_name"], "table_name": row["table_name"],
                "column_name": row["column_name"], "severity": "CRITICAL", "active": "true"}
        rules.append({**base, "rule_id": f"PK_NOT_NULL_{row['schema_name']}_{row['table_name']}_{row['column_name']}", "rule_type": "NOT_NULL"})
for (schema_name, table_name), key_columns in primary_keys.items():
    rules.append({"source_schema": schema_name, "table_name": table_name,
        "column_name": ",".join(key_columns), "severity": "CRITICAL", "active": "true",
        "rule_id": f"PK_UNIQUE_{schema_name}_{table_name}", "rule_type": "UNIQUE"})

# Contract-generated referential-integrity checks.
for row in schema_rows:
    if row.get("referenced_table") and row.get("referenced_column"):
        rules.append({
            "rule_id": f"FK_{row['schema_name']}_{row['table_name']}_{row['column_name']}",
            "active": "true", "severity": "ERROR", "rule_type": "REFERENTIAL_INTEGRITY",
            "source_schema": row["schema_name"], "table_name": row["table_name"],
            "column_name": row["column_name"], "referenced_schema": row.get("referenced_schema") or row["schema_name"],
            "referenced_table": row["referenced_table"], "referenced_column": row["referenced_column"],
        })

rules.extend(r.asDict() for r in custom_rules_df.where("lower(active) = 'true'").collect())
rule_fields = ["rule_id", "active", "severity", "rule_type", "source_schema", "table_name",
    "column_name", "referenced_schema", "referenced_table", "referenced_column",
    "operator", "rule_value", "description"]
normalised_rule_rows = [tuple(rule.get(field) for field in rule_fields) + (datetime.utcnow(),) for rule in rules]
spark.createDataFrame(normalised_rule_rows,
    "rule_id string,active string,severity string,rule_type string,source_schema string,table_name string,column_name string,referenced_schema string,referenced_table string,referenced_column string,operator string,rule_value string,description string,loaded_at timestamp") \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("monitoring.cfg_data_quality_rule")
print(f"Prepared {len(rules):,} data-quality rules")

In [ ]:
result_schema = "run_id string,rule_id string,severity string,rule_type string,source_table string,column_name string,status string,failed_row_count long,checked_row_count long,failure_percentage double,sample_key_json string,checked_at timestamp,message string"
reject_schema = "run_id string,rule_id string,source_table string,business_key_json string,rejection_reason string,rejected_at timestamp"
reference_schema = "run_id string,rule_id string,child_table string,child_column string,child_key string,parent_table string,parent_column string,detected_at timestamp"
result_rows = []
critical_failures = []

for rule in rules:
    rule_id = rule["rule_id"]
    rule_type = rule["rule_type"].upper()
    severity = (rule.get("severity") or "ERROR").upper()
    source = silver_table(rule["source_schema"], rule["table_name"])
    column_name = rule["column_name"]
    checked_at = datetime.utcnow()
    try:
        frame = spark.table(source)
        checked = frame.count()
        failed_frame = None
        failed = 0

        if rule_type == "NOT_NULL":
            failed_frame = frame.where(F.col(qident(column_name)).isNull())
            failed = failed_frame.count()
        elif rule_type == "UNIQUE":
            key_columns = [name.strip() for name in column_name.split(",") if name.strip()]
            non_null = frame
            for key_column in key_columns:
                non_null = non_null.where(F.col(qident(key_column)).isNotNull())
            duplicates = non_null.groupBy(*key_columns).count().where("count > 1")
            failed = duplicates.select(F.sum(F.col("count") - 1).alias("failed")).first()["failed"] or 0
            failed_frame = duplicates.select(F.to_json(F.struct(*[F.col(qident(c)) for c in key_columns])).alias("_key"))
        elif rule_type == "REFERENTIAL_INTEGRITY":
            parent = silver_table(rule["referenced_schema"], rule["referenced_table"])
            parent_frame = spark.table(parent).select(F.col(qident(rule["referenced_column"])).alias("_parent_key")).distinct()
            failed_frame = (frame.where(F.col(qident(column_name)).isNotNull())
                .select(F.col(qident(column_name)).cast("string").alias("_key")).distinct()
                .join(parent_frame, F.col("_key") == F.col("_parent_key"), "left_anti"))
            failed = failed_frame.count()
            refs = [(RUN_ID, rule_id, source, column_name, r["_key"], parent,
                rule["referenced_column"], checked_at) for r in failed_frame.limit(MAX_REJECT_REFERENCES_PER_RULE).collect()]
            append_rows("monitoring.cfg_referential_exception", refs, reference_schema)
        elif rule_type == "DATE_ORDER":
            other = rule["referenced_column"]
            failed_frame = frame.where(F.col(qident(column_name)).isNotNull() & F.col(qident(other)).isNotNull()
                & (F.col(qident(column_name)) > F.col(qident(other))))
            failed = failed_frame.count()
        elif rule_type == "NON_NEGATIVE":
            failed_frame = frame.where(F.col(qident(column_name)) < F.lit(0))
            failed = failed_frame.count()
        else:
            raise ValueError(f"Unsupported rule type: {rule_type}")

        status = "PASS" if failed == 0 else "FAIL"
        pct = (failed / checked * 100.0) if checked else 0.0
        sample_key = None
        if failed_frame is not None and failed:
            key_column = "_key" if "_key" in failed_frame.columns else column_name.split(",")[0].strip()
            samples = failed_frame.select(F.col(qident(key_column)).cast("string").alias("key")) \
                .limit(MAX_REJECT_REFERENCES_PER_RULE).collect()
            sample_key = samples[0]["key"] if samples else None
            rejects = [(RUN_ID, rule_id, source, '{"key":"' + str(r["key"]).replace('"', '\\"') + '"}',
                rule.get("description") or rule_type, checked_at) for r in samples]
            append_rows("monitoring.cfg_rejected_row", rejects, reject_schema)
        result_rows.append((RUN_ID, rule_id, severity, rule_type, source, column_name, status,
            int(failed), int(checked), float(pct), sample_key, checked_at, rule.get("description")))
        if status == "FAIL" and severity == "CRITICAL":
            critical_failures.append(rule_id)
    except Exception as exc:
        result_rows.append((RUN_ID, rule_id, severity, rule_type, source, column_name, "ERROR",
            0, 0, 0.0, None, checked_at, str(exc)[:2000]))
        if severity == "CRITICAL":
            critical_failures.append(rule_id)

append_rows("monitoring.cfg_data_quality_result", result_rows, result_schema)
failed_checks = sum(1 for row in result_rows if row[6] in ("FAIL", "ERROR"))
run_status = "FAILED" if critical_failures else ("SUCCESS_WITH_WARNINGS" if failed_checks else "SUCCESS")
spark.sql(f"""UPDATE monitoring.cfg_pipeline_run SET ended_at=current_timestamp(), status='{run_status}',
tables_succeeded={len(result_rows) - failed_checks}, tables_failed={failed_checks},
rows_read=0, rows_written={len(result_rows)}, error_message=NULL WHERE run_id='{RUN_ID}'""")
if critical_failures and FAIL_ON_CRITICAL:
    raise RuntimeError(f"Critical DQ failures: {critical_failures[:20]}")
print(f"DQ run {RUN_ID}: {len(result_rows)} checks; {len(critical_failures)} critical failures")